# 02 - Web tools and reading documents

**ICSSI 2026, Using LLMs via the API**

Two capabilities that show up in almost every science-of-science project. First, Claude
can search and fetch the web with server-side tools, so it can answer questions that need
current sources. Second, Claude can read documents you send it, such as a conference
program or a paper, and summarize them. We will do both, and track the cost of each.

In [1]:
# fetch the repo files (claude_kit.py and tutorial_data/) when running in Colab
import os, sys, subprocess
REPO_URL = "https://github.com/LarremoreLab/icssi-2026-hackathon.git"  # update after you push to GitHub

if "google.colab" in sys.modules and not os.path.exists("claude_kit.py"):
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir(REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git"))
print("working directory:", os.getcwd())
assert os.path.exists("claude_kit.py"), (
    "claude_kit.py was not found. Run this notebook from inside the cloned repo folder, "
    "or set REPO_URL above and run this cell again in Colab."
)

working directory: /Users/larremoreadmin/tutorials/anthropic-api


In [ ]:
# bootstrap: works in Google Colab and locally
import os, sys, subprocess

def pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

try:
    import anthropic
except ImportError:
    pip_install("anthropic")
    import anthropic

# find the api key: env var first, then a Colab secret, then a local .env file
if not os.environ.get("ANTHROPIC_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        try:
            from dotenv import load_dotenv
        except ImportError:
            pip_install("python-dotenv")
            from dotenv import load_dotenv
        load_dotenv()

assert os.environ.get("ANTHROPIC_API_KEY"), (
    "No ANTHROPIC_API_KEY found. Running locally, copy .env.example to .env and add your "
    "key. In Colab, add it in the Secrets panel (the key icon), named ANTHROPIC_API_KEY."
)
print("anthropic version:", anthropic.__version__, "| API key loaded: OK")
MODEL = "claude-sonnet-4-6"

anthropic version: 0.111.0 | API key loaded: OK


## Web tools run on the server

The `web_search` and `web_fetch` tools run on Anthropic's servers. You just declare them,
and Claude decides when to search. Results come back inside the same response, with
sources. 

Web search is billed extra per search at $10/1000 searches on top of tokens.
A single request can run several searches, so the `CostTracker` counts them for us.

Let us ask one current-events question.

**Warning:** LLM web search tools often seem to pull outdated versions of websites and thus can return outdated results. Use with caution!

In [3]:
import anthropic
from claude_kit import text_of, CostTracker

client = anthropic.Anthropic()
tracker = CostTracker(budget_usd=0.50)

response = client.messages.create(
    model=MODEL,
    max_tokens=600,
    tools=[{"type": "web_search_20260209", "name": "web_search"}],
    messages=[{"role": "user",
               "content": "Search the web: what is ICSSI, and where and when is the 2026 "
                          "conference held? Answer in two sentences and include a source URL."}],
)
tracker.add_response(model=MODEL, response=response, label="web-search")
print(text_of(response))

    # show what it actually searched for
for block in response.content:
    if getattr(block, "type", None) == "server_tool_use" and block.name == "web_search":
        print("\n[searched]", block.input.get("query"))

**ICSSI** (the International Conference on the Science of Science and Innovation) is an interdisciplinary event that brings together experts from various fields to explore the dynamics of scientific research and innovation, serving as a platform for both producers—scientists from academia and industry—and consumers, including policymakers, publishers, funders, and administrators. The 2026 edition will be held at the National Academy of Sciences in Washington, D.C., starting on June 29th, running for three days.

**Source:** https://icssi.org

[searched] ICSSI 2026 conference location date

[searched] ICSSI 2026 conference location date


## Reading a document: summarize Day 1 of the ICSSI program

You can send a PDF to the model as a document content block, alongside your text prompt.
The model reads the pages and you ask questions about them. PDFs can be large and every
page costs input tokens, so a good habit is to send only the pages you need.

Here we use `pypdf` to slice pages 3 to 5 out of the real ICSSI 2026 program,
`tutorial_data/ICSSI_2026_Program_Full_v7.pdf`. Those pages are Day 1 (Monday June 29): the morning
and afternoon talk sessions on pages 3 and 4, and the start of Poster Session I on page 5.
We ask the model to summarize the themes of the Day 1 talks and the themes of the Day 1
posters separately. This is a small example of a document-summarization task you could run
over many program PDFs, syllabi, or papers.

In [ ]:
import base64, io
from pypdf import PdfReader, PdfWriter

# slice pages 3 to 6 (Day 1 talks and the start of the poster session) into a small pdf.
# pypdf may print harmless "Ignoring wrong pointing object" warnings for some pdfs.
program_pdf = "tutorial_data/ICSSI_2026_Program_Full_v7.pdf"
reader = PdfReader(program_pdf)
writer = PdfWriter()
for page_index in range(2, 6):           # zero-indexed pages 3-6
    writer.add_page(reader.pages[page_index])
buffer = io.BytesIO()
writer.write(buffer)
pdf_base64 = base64.standard_b64encode(buffer.getvalue()).decode()

pdf_response = client.messages.create(
    model=MODEL,
    max_tokens=700,
    messages=[{
        "role": "user",
        "content": [
            {"type": "document",
             "source": {"type": "base64", "media_type": "application/pdf", "data": pdf_base64}},
            {"type": "text",
             "text": "This document is the schedule of Day 1 of the ICSSI academic conference. "
                     "Summarize the main themes of the talks and poster session in MAXIMUM 3 bullet points."},
        ],
    }],
)
tracker.add_response(model=MODEL, response=pdf_response, label="pdf-summary")
print(text_of(pdf_response))

Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 19 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 23 0 (offset 0)
Ignoring wrong pointing object 25 0 (offset 0)
Ignoring wrong pointing object 27 0 (offset 0)
Ignoring wrong pointing object 29 0 (offset 0)
Ignoring wrong pointing object 31 0 (offset 0)
Ignoring wrong pointing object 33 0 (offset 0)
Ignoring wrong pointing object 35 0 (offset 0)
Ignoring wrong pointing object 49 0 (offset 0)
Ignoring wrong pointing object 54 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)
Ignoring wrong pointing object 61 0 (offset 0)
Ignoring wrong pointing object 63 0 (offset 0)
Ignoring wrong pointing object 65 0 (offset 0)
Ignoring wrong 

# ICSSI 2026 Day 1 – Summary

## INVITED TALKS & PANEL DISCUSSION THEMES

- **AI's Impact on Science**: How artificial intelligence (particularly tools like AlphaFold) is reshaping scientific research and reorganizing the research ecosystem

- **Scientific Workforce & Career Dynamics**: Talent mobility, mentorship disparities across racial lines, researcher productivity, leadership's influence on scientific direction, and delayed recognition (e.g., Nobel Prize timing)

- **Social Contract for Science**: Historical and future perspectives on the relationship between science and society, including risk-reward considerations for scholars

- **Research Integrity & Ethics**: Overview of national science funding initiatives and their activities in the science of science

## CONTRIBUTED TALKS & PANELS THEMES

- **AI & Large Language Models**: Human-AI collaboration, LLM integration in clinical trials, improving AI for scientific discovery, and LLM-assisted writing benefits

- **Career Traject

## Cost of this session

The web search call is billed per search on top of tokens, and the PDF call is billed by
the tokens the pages take up. Here is the breakdown.

In [6]:
tracker.report()

model                calls    in tok   out tok    cached  searches      cost $
------------------------------------------------------------------------------
claude-sonnet-4-6        1     30652       597         0         2      0.1209
claude-haiku-4-5         1     10803       628         0         0      0.0139
------------------------------------------------------------------------------
TOTAL                    2                                       2      0.1349


## Recap

Web tools, namely `web_search` and `web_fetch`, run on the server, so you just declare
them and Claude uses them when needed. Web search has a per-search charge that the
`CostTracker` records for you. Sending a PDF as a document block lets Claude read and
summarize files, and slicing to only the pages you need keeps the token cost down.

The take-home notebooks go further. Notebook 11 covers structured output and your own
tools. Notebook 12 covers interactive chat compared with unattended agents. Notebook 99 is
a capstone template you can fork for your project.